# 02 数据读写：CSV / Excel / JSON / Parquet

真实项目里，90% 的 pandas 问题都从"读进来"开始：

- 读进来全是字符串，没法算数；
- 日期读成了文本，没法按时间聚合；
- 中文乱码、`utf-8` 和 `gbk` 选错；
- 大文件一次性读爆内存；
- 读写一轮下来，类型丢了、索引多了一列。

本节用 `data/` 下的真实样例文件，把这些坑一个个踩一遍。

> 运行环境：`llm_sft`，请从 `python/pandas` 目录启动 Jupyter（这样 `data/` 能直接找到）。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
if not DATA.exists():                       # 从仓库根目录打开时的兜底
    DATA = Path("python/pandas/data")
BASE = DATA.parent
OUT = BASE / "output"
OUT.mkdir(exist_ok=True)

print("数据目录:", DATA.resolve())
for p in sorted(DATA.rglob("*")):
    if p.is_file() and p.suffix in {".csv", ".json", ".jsonl", ".parquet", ".xlsx", ".md", ".txt", ".html"}:
        print(f"  {p.relative_to(DATA).as_posix():28s} {p.stat().st_size:>7,} 字节")

数据目录: D:\Repository\Jianyu-is-learning\python\pandas\data
  customer_orders.csv              399 字节
  customers.csv                    433 字节
  customers.json                 1,084 字节
  customers_gbk.csv                 66 字节
  docs/产品说明.md                   1,446 字节
  docs/客服问答.txt                  1,129 字节
  docs/技术博客.html                 1,908 字节
  docs/转载_客服问答.txt                 696 字节
  events.jsonl                   5,193 字节
  prices_wide.csv                  203 字节
  raw_sft/alpaca_raw.jsonl       1,542 字节
  raw_sft/chat_logs.jsonl        5,320 字节
  raw_sft/faq_qa.json            1,243 字节
  sales_orders.csv               3,689 字节
  sales_orders_messy.csv           666 字节
  site_visits.csv               14,356 字节


## 1. `read_csv`：先看原始文本，再谈参数

读文件最稳的习惯：**先用 `read_text()` 看一眼前几行**，确认分隔符、表头、编码，
再决定 `read_csv` 的参数，比反复猜快得多。

In [2]:
raw = (DATA / "sales_orders_messy.csv").read_text(encoding="utf-8")
print(raw[:420])

order_id,order_date,city,category,unit_price,qty,amount,sales_rep
S001,2024-01-05, 北京 ,数码,"1,299.00",2,2598.00,张伟
S002,2024/01/07,上海,家居,199.5,1,199.5, 李娜 
S003,20240109,广州,服饰,¥ 89.9,3,269.7,王强
S004,2024-02-30,深圳,食品,12.5,,12.5,赵敏
S005,2024-03-15 10:20,杭州,数码,899,2,1798,陈晨
S006,2024-03-16,北京,digital,1299,1,1299,张伟
S007,2024-03-18,成都,食品,9.9,-1,-9.9,刘洋
S007,2024-03-18,成都,食品,9.9,-1,-9.9,刘洋
,2024-04-01,,家居,299,1,299,李娜
S009


In [3]:
orders = pd.read_csv(DATA / "sales_orders_messy.csv")
orders

,order_id,order_date,city,category,unit_price,qty,amount,sales_rep
0,S001,2024-01-05,北京,数码,"1,299.00",2.0,2598.0,张伟
1,S002,2024/01/07,上海,家居,199.5,1.0,199.5,李娜
2,S003,20240109,广州,服饰,¥ 89.9,3.0,269.7,王强
3,S004,2024-02-30,深圳,食品,12.5,NaN,12.5,赵敏
4,S005,2024-03-15 10:20,杭州,数码,899,2.0,1798.0,陈晨
5,S006,2024-03-16,北京,digital,1299,1.0,1299.0,张伟
6,S007,2024-03-18,成都,食品,9.9,-1.0,-9.9,刘洋
7,S007,2024-03-18,成都,食品,9.9,-1.0,-9.9,刘洋
8,NaN,2024-04-01,NaN,家居,299,1.0,299.0,李娜
9,S009,2024-04-05,上海,服饰,89.9,0.0,0.0,王强


In [4]:
# 自动推断的结果：能变数字的都变了，但日期和"带符号的金额"还是文本
orders.dtypes

order_id          str
order_date        str
city              str
category          str
unit_price        str
qty           float64
amount        float64
sales_rep         str
dtype: object

### 常用参数

| 参数 | 作用 | 本文件的用法 |
| --- | --- | --- |
| `sep` | 分隔符 | `sep=","`（默认）或 `"\t"` |
| `header` | 表头行号 | `header=None` 表示文件没有表头 |
| `names` | 自己指定列名 | `names=["a", "b"]` |
| `usecols` | 只读部分列 | 大表省内存 |
| `nrows` | 只读前 N 行 | 快速预览 |
| `dtype` | 指定列类型 | 防止 ID 被转成数字 |
| `na_values` | 哪些值算缺失 | `["", "N/A", "-"]` |
| `parse_dates` | 解析日期列 | 要求格式统一 |
| `encoding` | 文件编码 | `utf-8` / `gbk` / `utf-8-sig` |

下面这些坑在真实数据里很常见：

In [5]:
# 坑 1：金额列带千分位和货币符号 → 整列变 str
print("unit_price 的 dtype:", orders["unit_price"].dtype)
print("前 3 个值:", orders["unit_price"].head(3).tolist())

# 坑 2：订单号、身份证、手机号这类"数字"必须按字符串读，否则前导 0 会丢
ids = pd.read_csv(DATA / "sales_orders.csv", usecols=["order_id"], dtype={"order_id": "string"})
print("order_id dtype:", ids["order_id"].dtype, "| 前 3 个:", ids["order_id"].head(3).tolist())

unit_price 的 dtype: str
前 3 个值: ['1,299.00', '199.5', '¥ 89.9']
order_id dtype: string | 前 3 个: ['O001', 'O002', 'O003']


In [6]:
# 坑 3：parse_dates 遇到混合日期格式会"静默失效"——不报错，但列还是字符串
parsed = pd.read_csv(DATA / "sales_orders_messy.csv", parse_dates=["order_date"])
print("order_date 的 dtype:", parsed["order_date"].dtype)   # str 说明根本没解析成功

# 正确做法：先用字符串读进来，再统一转换
# format="mixed" 允许每行格式不同；errors="coerce" 把非法日期（2024-02-30）变成 NaT
fixed = pd.to_datetime(parsed["order_date"], format="mixed", errors="coerce")
print("转换后 dtype:", fixed.dtype)
print(fixed.tolist())

order_date 的 dtype: str
转换后 dtype: datetime64[us]
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-09 00:00:00'), NaT, Timestamp('2024-03-15 10:20:00'), Timestamp('2024-03-16 00:00:00'), Timestamp('2024-03-18 00:00:00'), Timestamp('2024-03-18 00:00:00'), Timestamp('2024-04-01 00:00:00'), Timestamp('2024-04-05 00:00:00'), NaT, Timestamp('2024-05-20 00:00:00')]


> **教训**：`parse_dates` 不保证成功，也从不报错。读完一定 `print(df.dtypes)`，
> 日期列不是 `datetime64` 就说明没解析成功，回到 `pd.to_datetime(..., errors="coerce")` 手动处理。

In [7]:
# 常用参数组合：只读需要的列、只读前几行、把垃圾值识别为缺失
peek = pd.read_csv(
    DATA / "sales_orders_messy.csv",
    usecols=["order_id", "order_date", "city", "qty"],
    nrows=5,
    na_values=["", "N/A", "None"],
)
peek

,order_id,order_date,city,qty
0,S001,2024-01-05,北京,2.0
1,S002,2024/01/07,上海,1.0
2,S003,20240109,广州,3.0
3,S004,2024-02-30,深圳,NaN
4,S005,2024-03-15 10:20,杭州,2.0


In [8]:
# 文件没有表头：header=None + names 自己命名（这里跳过原来那行表头）
no_header = pd.read_csv(
    DATA / "prices_wide.csv",
    skiprows=1,
    header=None,
    names=["产品", "2021", "2022", "2023", "2024"],
)
no_header

,产品,2021,2022,2023,2024
0,智能音箱,299.0,279.0,259.0,249.0
1,无线耳机,499.0,469.0,429.0,399.0
2,扫地机器人,1999.0,1899.0,1799.0,1699.0
3,智能台灯,129.0,119.0,109.0,99.0


## 2. 编码：中文乱码的头号原因

- `utf-8`：最通用，绝大多数新文件都是它；
- `gbk` / `gb18030`：Windows 上 Excel"另存为 CSV"的默认编码，中文必乱；
- `utf-8-sig`：带 BOM 的 UTF-8，**写出给 Excel 打开时最省事**。

判断方法：读出来是大段 `锟斤拷` / `Ã¤Â¸Â` → 编码选错；`UnicodeDecodeError` → 也是选错。

In [9]:
try:
    pd.read_csv(DATA / "customers_gbk.csv", encoding="utf-8")
except UnicodeDecodeError as e:
    print("用 utf-8 读 gbk 文件失败 →", str(e)[:90])

gbk_df = pd.read_csv(DATA / "customers_gbk.csv", encoding="gbk")
gbk_df

用 utf-8 读 gbk 文件失败 → 'utf-8' codec can't decode byte 0xd0 in position 0: invalid continuation byte


,姓名,城市,备注
0,张三,北京,老客户
1,李四,上海,NaN
2,王五,广州,需要发票


In [10]:
# 写出给 Excel 打开：utf-8-sig（带 BOM），否则 Excel 会显示乱码
gbk_df.to_csv(OUT / "customers_for_excel.csv", index=False, encoding="utf-8-sig")
print((OUT / "customers_for_excel.csv").read_text(encoding="utf-8-sig"))

姓名,城市,备注
张三,北京,老客户
李四,上海,
王五,广州,需要发票



## 3. 大文件：分块读（`chunksize`）

文件比内存大时，不要硬读。`chunksize` 返回一个**迭代器**，每块是一个小 DataFrame，
在块内聚合，再合并结果——这就是"分而治之"。

In [11]:
chunk_stats = []
for chunk in pd.read_csv(DATA / "site_visits.csv", chunksize=60):
    agg = chunk.groupby("page")["duration_s"].agg(["count", "sum"])
    chunk_stats.append(agg)

# 每个块的结果按 page 再汇总一次
page_stats = pd.concat(chunk_stats).groupby(level=0).sum().sort_values("sum", ascending=False)
page_stats

,count,sum
page,,
/blog,90,29195
/pricing,96,27970
/home,81,24094
/docs,75,21620


## 4. Excel：一表多 sheet 的正确姿势

需要 `openpyxl`（`llm_sft` 环境已装好）。`sheet_name=None` 读全部 sheet 返回字典。

In [12]:
orders_clean = pd.read_csv(DATA / "sales_orders.csv")
sheet_path = OUT / "report.xlsx"

with pd.ExcelWriter(sheet_path, engine="openpyxl") as writer:
    orders_clean.head(20).to_excel(writer, sheet_name="明细", index=False)
    orders_clean.groupby("city")["amount"].sum().to_excel(writer, sheet_name="按城市汇总")
print("已写出:", sheet_path.name, sheet_path.stat().st_size, "字节")

sheets = pd.read_excel(sheet_path, sheet_name=None)
print("sheet 列表:", list(sheets))
sheets["按城市汇总"].head()

已写出: report.xlsx 6766 字节
sheet 列表: ['明细', '按城市汇总']


,city,amount
0,上海,9723.9
1,北京,9986.5
2,广州,10123.6
3,成都,14646.7
4,杭州,15879.8


## 5. JSON 与 JSONL：非结构化数据最常用的格式

- **JSON**：整个文件是一个对象/数组，适合配置、接口返回；
- **JSONL / NDJSON**：一行一条记录，适合日志、SFT 数据——能流式读写、坏一行不影响全文。

嵌套结构有两个关键工具：`pd.json_normalize()`（把嵌套字典展平）和 `explode()`（把列表拆成多行）。

In [13]:
import json

customers = pd.read_json(DATA / "customers.json")
customers

,id,name,city,vip,tags,contact,created_at
0,C01,北京晨光科技,北京,True,"[企业, 长期合作]","{'email': 'contact@chenguang.example.com', 'ph...",2022-03-01 09:30:00
1,C02,上海蓝鲸贸易,上海,False,[外贸],"{'email': 'hi@lanjing.example.com', 'phone': N...",2022-07-15 14:05:00
2,C03,广州南风服饰,广州,False,None,NaN,2023-01-09 10:00:00
3,C04,深圳千里数码,深圳,True,"[企业, 数码, 高价值]","{'email': 'sales@qianli.example.com', 'phone':...",2021-11-20 16:45:00


In [14]:
# 展平嵌套字典：contact.email / contact.phone 变成独立列
flat = pd.json_normalize(customers.to_dict("records"))
flat

,id,name,city,vip,tags,created_at,contact.email,contact.phone,contact
0,C01,北京晨光科技,北京,True,"[企业, 长期合作]",2022-03-01 09:30:00,contact@chenguang.example.com,010-8888-1234,NaN
1,C02,上海蓝鲸贸易,上海,False,[外贸],2022-07-15 14:05:00,hi@lanjing.example.com,NaN,NaN
2,C03,广州南风服饰,广州,False,None,2023-01-09 10:00:00,NaN,NaN,NaN
3,C04,深圳千里数码,深圳,True,"[企业, 数码, 高价值]",2021-11-20 16:45:00,sales@qianli.example.com,0755-6666-0000,NaN


In [15]:
# 列表字段：tags 是一个 list，explode 可以拆成多行
exploded = flat.explode("tags")
exploded[["id", "name", "tags"]]

,id,name,tags
0,C01,北京晨光科技,企业
0,C01,北京晨光科技,长期合作
1,C02,上海蓝鲸贸易,外贸
2,C03,广州南风服饰,NaN
3,C04,深圳千里数码,企业
3,C04,深圳千里数码,数码
3,C04,深圳千里数码,高价值


In [16]:
# JSONL：一行一条 JSON，lines=True
events = pd.read_json(DATA / "events.jsonl", lines=True)
print("行数:", len(events), "| props 列类型:", type(events.loc[0, "props"]).__name__)
events.head(3)

行数: 40 | props 列类型: dict


,ts,user_id,event,props
0,2024-06-04T04:01:00,u1003,add_cart,"{'page': '/help', 'stay_s': 173, 'device': 'web'}"
1,2024-06-03T20:01:00,u1001,view,"{'page': '/product/123', 'stay_s': 86, 'device..."
2,2024-06-04T01:42:00,u1000,click,"{'page': '/help', 'stay_s': 40, 'device': 'web'}"


In [17]:
# 把 props 展平后再分析
events_flat = pd.json_normalize(events.to_dict("records"))
events_flat.groupby("props.page")["props.stay_s"].agg(["count", "mean"]).round(1)

,count,mean
props.page,,
/cart,9,138.7
/checkout,7,127.7
/help,10,151.6
/home,7,141.0
/product/123,6,191.7


## 6. Parquet / Feather：数据工程的默认选择

| 特性 | CSV | Parquet |
| --- | --- | --- |
| 体积 | 大（文本、无压缩） | 小（列式 + 压缩，通常 1/5 ~ 1/10） |
| 速度 | 慢（要解析文本） | 快（列式、只读需要的列） |
| 类型 | 全部是文本，读回要重新推断 | **dtype 原样保留**（日期、category、可空整数） |
| 人类可读 | 是 | 否（需要工具打开） |

结论：**中间结果一律存 Parquet，最终交付给人看时才导出 CSV / Excel。**

In [18]:
# 先验证"dtype 原样保留"：日期、可空整数、category 都不会在往返中丢失
orders_clean.to_parquet(OUT / "orders.parquet", compression="zstd")
roundtrip = pd.read_parquet(OUT / "orders.parquet")
print("读回后 dtype 与写出前一致:", roundtrip.dtypes.equals(orders_clean.dtypes))
roundtrip.dtypes

读回后 dtype 与写出前一致: True


order_id          str
order_date        str
city              str
category          str
channel           str
unit_price    float64
qty             int64
amount        float64
sales_rep         str
dtype: object

In [19]:
# 再看体积和速度：把 60 行放大成 3 万行，模拟真实项目规模
import time

rng = np.random.default_rng(0)
big = pd.concat([orders_clean] * 500, ignore_index=True)
big["order_id"] = "X" + big.index.astype(str)                    # 让每行唯一
big["amount"] = (big["amount"] + rng.normal(0, 5, len(big))).round(2)

big.to_csv(OUT / "big.csv", index=False)
big.to_parquet(OUT / "big.parquet", compression="zstd")

csv_size = (OUT / "big.csv").stat().st_size
pq_size = (OUT / "big.parquet").stat().st_size
print(f"行数    : {len(big):,}")
print(f"CSV     : {csv_size:>10,} 字节")
print(f"Parquet : {pq_size:>10,} 字节   （约为 CSV 的 1/{csv_size / pq_size:.1f}）")

t0 = time.perf_counter()
pd.read_csv(OUT / "big.csv")
t1 = time.perf_counter()
pd.read_parquet(OUT / "big.parquet")
t2 = time.perf_counter()
print(f"读取耗时: CSV {t1 - t0:.2f}s | Parquet {t2 - t1:.2f}s")

行数    : 30,000
CSV     :  1,823,145 字节
Parquet :    212,298 字节   （约为 CSV 的 1/8.6）


读取耗时: CSV 0.05s | Parquet 0.01s

## 7. 写出 CSV 的参数

- `index=False`：99% 的情况都该加，否则会多一列 `Unnamed: 0`；
- `na_rep`：缺失值写成什么（默认空字符串）；
- `float_format`：控制小数位；
- `columns`：只写部分列、调整顺序；
- `encoding="utf-8-sig"`：给 Excel 看。

In [20]:
orders_clean.head(5).to_csv(
    OUT / "orders_delivery.csv",
    index=False,
    encoding="utf-8-sig",
    na_rep="-",
    float_format="%.2f",
    columns=["order_id", "order_date", "city", "amount"],
)
print((OUT / "orders_delivery.csv").read_text(encoding="utf-8-sig"))

order_id,order_date,city,amount
O001,2024-01-08,上海,383.60
O002,2024-01-12,广州,1813.60
O003,2024-01-13,杭州,362.10
O004,2024-01-16,北京,3033.60
O005,2024-01-17,广州,403.00



## 8. 格式怎么选

| 场景 | 推荐格式 |
| --- | --- |
| 别人发来的原始数据 | 先按原样读，读完立刻转 Parquet 存档 |
| 中间结果、训练数据 | Parquet（JSONL 用于流式/逐条处理） |
| 交付给业务方看 | Excel（带 sheet）或 CSV |
| 配置、接口返回 | JSON |
| 日志、SFT 语料 | JSONL（一行一条，容错好） |

## 9. 小练习

把 `sales_orders_messy.csv` 读进来，做三件事：

1. 用 `dtype` 把 `order_id` 读成字符串，`qty` 读成可空整数 `Int64`；
2. 把 `unit_price` 里的 `¥`、`,`、空格去掉，转成 `float`；
3. 把清洗后的结果写成 Parquet，并打印"原始行数 / 去重后行数 / 缺失值最多的列"。

In [21]:
student_orders = pd.read_csv(
    DATA / "sales_orders_messy.csv",
    dtype={"order_id": "string", "qty": "Int64"},
    na_values=["", "N/A"],
)

price = (
    student_orders["unit_price"]
    .astype("string")
    .str.replace("¥", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
student_orders["unit_price"] = pd.to_numeric(price, errors="coerce")

print("原始行数      :", len(student_orders))
print("完整重复行数  :", student_orders.duplicated().sum())
print("缺失值最多的一列:", student_orders.isna().sum().idxmax())

student_orders.to_parquet(OUT / "student_orders.parquet", index=False)
student_orders.head()

原始行数      : 12
完整重复行数  : 1
缺失值最多的一列: order_id


,order_id,order_date,city,category,unit_price,qty,amount,sales_rep
0,S001,2024-01-05,北京,数码,1299.0,2,2598.0,张伟
1,S002,2024/01/07,上海,家居,199.5,1,199.5,李娜
2,S003,20240109,广州,服饰,89.9,3,269.7,王强
3,S004,2024-02-30,深圳,食品,12.5,<NA>,12.5,赵敏
4,S005,2024-03-15 10:20,杭州,数码,899.0,2,1798.0,陈晨


## 10. 小结

1. 读之前先 `read_text()` 看几行，确认分隔符、表头、编码。
2. `dtype` 是防坑第一参数：ID 用字符串，缺失整数用 `Int64`。
3. 日期列格式不统一时，`parse_dates` 会失败，改用 `pd.to_datetime(..., errors="coerce")`。
4. `chunksize` 处理超大文件；块内聚合、再合并。
5. `openpyxl` + `ExcelWriter` 写多 sheet；`sheet_name=None` 一次读全部。
6. `json_normalize` 展平字典，`explode` 展开列表，`lines=True` 读 JSONL。
7. 中间结果用 Parquet：小、快、dtype 不丢。
8. 写 CSV 记得 `index=False`，给 Excel 看就加 `encoding="utf-8-sig"`。